In [ ]:
!git clone https://github.com/sumit760/Codeglue.git

In [2]:
import os

# Create the clean Dev/SRC directory path inside Kaggle
os.makedirs("Dev/SRC", exist_ok=True)
print("✅ Directory 'Dev/SRC' created successfully!")

✅ Directory 'Dev/SRC' created successfully!


In [4]:
%%writefile Dev/SRC/data_preprocessing.py
import os
import json
from datasets import load_dataset
from transformers import AutoTokenizer

def clean_and_tokenize():
    print("--- STEP 1: LOADING DATASET ---")
    dataset = load_dataset("google/code_x_glue_cc_defect_detection")
    
    print("\n--- STEP 2: LOADING TOKENIZER ---")
    model_name = "huggingface/CodeBERTa-small-v1"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    print("\n--- STEP 3: PREPROCESSING DATA ---")
    def preprocess_function(examples):
        cleaned_code = [" ".join(code.split()) for code in examples["func"]]
        model_inputs = tokenizer(
            cleaned_code,
            truncation=True,
            padding="max_length",
            max_length=512
        )
        model_inputs["labels"] = [1 if label else 0 for label in examples["target"]]
        return model_inputs

    columns_to_remove = ['id', 'func', 'target', 'project', 'commit_id']
    tokenized_dataset = dataset.map(
        preprocess_function, 
        batched=True, 
        remove_columns=columns_to_remove
    )
    
    print("\n--- STEP 4: SAVING ARTIFACTS ---")
    output_dir = "./processed_dataset"
    tokenized_dataset.save_to_disk(output_dir)
    print(f"Dataset successfully processed and saved to '{output_dir}'")
    
    id2label = {0: "Safe", 1: "Defective"}
    with open("id2label.json", "w") as f:
        json.dump(id2label, f)
    print("id2label.json file saved successfully.")

if __name__ == "__main__":
    clean_and_tokenize()

Overwriting Dev/SRC/data_preprocessing.py


In [4]:
import os

# 1. Define absolute paths
base_dir = "/kaggle/working/Dev/SRC"
file_path = os.path.join(base_dir, "data_preprocessing.py")

# 2. Ensure directories exist
os.makedirs(base_dir, exist_ok=True)

# 3. Code content to write
code_content = """import os
import json
from datasets import load_dataset
from transformers import AutoTokenizer

def clean_and_tokenize():
    print("--- STEP 1: LOADING DATASET ---")
    dataset = load_dataset("google/code_x_glue_cc_defect_detection")
    
    print("\\n--- STEP 2: LOADING TOKENIZER ---")
    model_name = "huggingface/CodeBERTa-small-v1"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    print("\\n--- STEP 3: PREPROCESSING DATA ---")
    def preprocess_function(examples):
        cleaned_code = [" ".join(code.split()) for code in examples["func"]]
        model_inputs = tokenizer(
            cleaned_code,
            truncation=True,
            padding="max_length",
            max_length=512
        )
        model_inputs["labels"] = [1 if label else 0 for label in examples["target"]]
        return model_inputs

    columns_to_remove = ['id', 'func', 'target', 'project', 'commit_id']
    tokenized_dataset = dataset.map(
        preprocess_function, 
        batched=True, 
        remove_columns=columns_to_remove
    )
    
    print("\\n--- STEP 4: SAVING ARTIFACTS ---")
    output_dir = "/kaggle/working/processed_dataset"
    tokenized_dataset.save_to_disk(output_dir)
    print(f"Dataset successfully processed and saved to '{output_dir}'")
    
    id2label = {0: "Safe", 1: "Defective"}
    with open("/kaggle/working/id2label.json", "w") as f:
        json.dump(id2label, f)
    print("id2label.json file saved successfully.")

if __name__ == "__main__":
    clean_and_tokenize()
"""

# 4. Write the file safely
with open(file_path, "w") as f:
    f.write(code_content)

# 5. Double check file existence
if os.path.exists(file_path):
    print(f"✅ Success! File physically exists at: {file_path}")
    print("Contents of Dev/SRC directory:", os.listdir(base_dir))
else:
    print("❌ Error: File was not created.")

✅ Success! File physically exists at: /kaggle/working/Dev/SRC/data_preprocessing.py
Contents of Dev/SRC directory: ['data_preprocessing.py']


In [5]:
!python3 /kaggle/working/Dev/SRC/data_preprocessing.py

--- STEP 1: LOADING DATASET ---
README.md: 5.60kB [00:00, 2.88MB/s]
data/train-00000-of-00001.parquet: 100%|███| 17.8M/17.8M [00:01<00:00, 9.21MB/s]
data/validation-00000-of-00001.parquet: 100%|█| 2.21M/2.21M [00:01<00:00, 2.19MB
data/test-00000-of-00001.parquet: 100%|████| 2.23M/2.23M [00:00<00:00, 3.64MB/s]
Generating train split: 100%|██| 21854/21854 [00:00<00:00, 117918.53 examples/s]
Generating validation split: 100%|█| 2732/2732 [00:00<00:00, 160285.89 examples/
Generating test split: 100%|█████| 2732/2732 [00:00<00:00, 153816.11 examples/s]

--- STEP 2: LOADING TOKENIZER ---
config.json: 100%|█████████████████████████████| 480/480 [00:00<00:00, 3.13MB/s]
tokenizer_config.json: 100%|██████████████████| 19.0/19.0 [00:00<00:00, 133kB/s]
vocab.json: 994kB [00:00, 36.3MB/s]
merges.txt: 483kB [00:00, 98.1MB/s]

--- STEP 3: PREPROCESSING DATA ---
Map: 100%|█████████████████████████| 2732/2732 [00:01<00:00, 1584.27 examples/s]

--- STEP 4: SAVING ARTIFACTS ---
Saving the dataset (1/1 sh

In [6]:
import os

# 1. Define absolute paths
base_dir = "/kaggle/working/Dev/SRC"
file_path = os.path.join(base_dir, "train.py")

# 2. Training code content
code_content = """import os
import json
import numpy as np
import evaluate
from datasets import load_from_disk
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    AutoTokenizer
)
import wandb
from kaggle_secrets import UserSecretsClient

def train():
    print("--- STEP 1: AUTHENTICATING W&B ---")
    try:
        secrets = UserSecretsClient()
        wandb_key = secrets.get_secret("WANDB_API_KEY")
        os.environ["WANDB_API_KEY"] = wandb_key
        wandb.login()
    except Exception as e:
        print("⚠️ W&B Secret not found or login failed. Training will continue locally.")

    print("\\n--- STEP 2: LOADING PROCESSED DATASET ---")
    dataset_path = "/kaggle/working/processed_dataset"
    tokenized_dataset = load_from_disk(dataset_path)
    
    print("\\n--- STEP 3: LOADING MODEL & CONFIG ---")
    model_name = "huggingface/CodeBERTa-small-v1"
    
    with open("/kaggle/working/id2label.json", "r") as f:
        id2label = json.load(f)
    id2label = {int(k): v for k, v in id2label.items()}
    label2id = {v: k for k, v in id2label.items()}

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2,
        id2label=id2label,
        label2id=label2id
    )
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    print("\\n--- STEP 4: DEFINING EVALUATION METRICS ---")
    accuracy_metric = evaluate.load("accuracy")
    f1_metric = evaluate.load("f1")

    def compute_metrics(eval_pred):
        predictions, labels = eval_pred
        preds = np.argmax(predictions, axis=1)
        acc = accuracy_metric.compute(predictions=preds, references=labels)["accuracy"]
        f1 = f1_metric.compute(predictions=preds, references=labels, average="binary")["f1"]
        return {"eval_accuracy": acc, "eval_f1": f1}

    print("\\n--- STEP 5: SETTING TRAINING ARGUMENTS ---")
    # Utilizing your optimal Version 1 parameters
    training_args = TrainingArguments(
        output_dir="/kaggle/working/results",
        learning_rate=3e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=3,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_f1",
        fp16=True,
        report_to="wandb",
        logging_steps=50
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["validation"],
        tokenizer=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
        compute_metrics=compute_metrics,
    )

    print("\\n--- STEP 6: EXECUTING MODEL FINE-TUNING ---")
    trainer.train()
    
    print("\\n--- STEP 7: EXPORTING BEST MODEL WEIGHTS ---")
    final_model_dir = "/kaggle/working/best_model"
    trainer.save_model(final_model_dir)
    print(f"✅ Best model weights successfully saved to: {final_model_dir}")

if __name__ == "__main__":
    train()
"""

# 3. Write the script safely
with open(file_path, "w") as f:
    f.write(code_content)

# 4. Verify file generation
if os.path.exists(file_path):
    print(f"✅ Success! File physically exists at: {file_path}")
    print("Contents of Dev/SRC directory:", os.listdir(base_dir))
else:
    print("❌ Error: File was not created.")

✅ Success! File physically exists at: /kaggle/working/Dev/SRC/train.py
Contents of Dev/SRC directory: ['data_preprocessing.py', 'train.py']


In [9]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.1 MB/s eta 0:00:00


In [11]:
import os

file_path = "/kaggle/working/Dev/SRC/train.py"

# Read the current script
with open(file_path, "r") as f:
    code = f.read()

# Replace the outdated argument with the modern equivalent
code = code.replace("tokenizer=tokenizer,", "processing_class=tokenizer,")

# Write the fixed code back to the file
with open(file_path, "w") as f:
    f.write(code)

print("✅ Successfully updated train.py to match the latest Hugging Face API!")

✅ Successfully updated train.py to match the latest Hugging Face API!


In [12]:
!python3 /kaggle/working/Dev/SRC/train.py

--- STEP 1: AUTHENTICATING W&B ---
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: prudhvigowroju-work (prudhvigowroju_iitj) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin

--- STEP 2: LOADING PROCESSED DATASET ---

--- STEP 3: LOADING MODEL & CONFIG ---
Loading weights: 100%|█| 101/101 [00:00<00:00, 2220.71it/s, Materializing param=
RobertaForSequenceClassification LOAD REPORT from: huggingface/CodeBERTa-small-v1
Key                         | Status     | 
----------------------------+------------+-
lm_head.decoder.bias        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.decoder.weight      | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
class

In [2]:
import os

# 1. Define absolute paths
base_dir = "/kaggle/working/Dev/SRC"
file_path = os.path.join(base_dir, "inference.py")

# 1.5 FORCE DIRECTORY CREATION (Fix for FileNotFoundError)
os.makedirs(base_dir, exist_ok=True)

# 2. Inference code content
code_content = """import torch
import json
from transformers import AutoModelForSequenceClassification, AutoTokenizer

def run_inference(code_snippet):
    print("--- STEP 1: LOADING SAVED MODEL & TOKENIZER ---")
    # Path where your best model was saved
    model_path = "/kaggle/working/best_model"
    # Tokenizer is loaded from the base model name
    base_model_name = "huggingface/CodeBERTa-small-v1"
    
    try:
        model = AutoModelForSequenceClassification.from_pretrained(model_path)
        tokenizer = AutoTokenizer.from_pretrained(base_model_name)
    except Exception as e:
        print(f"❌ Error loading model: {e}")
        return

    print("\\n--- STEP 2: PREPARING INPUT DATA ---")
    # Clean the input code exactly as we did in preprocessing
    cleaned_code = " ".join(code_snippet.split())
    
    inputs = tokenizer(
        cleaned_code,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=512
    )

    print("\\n--- STEP 3: RUNNING PREDICTION ---")
    # Run the model in evaluation mode
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        
    # Get the predicted class index
    logits = outputs.logits
    predicted_class_id = logits.argmax(dim=1).item()
    
    # Map index back to label
    id2label = {0: "Safe", 1: "Defective"}
    prediction = id2label[predicted_class_id]
    
    print("\\n==================================")
    print("      INFERENCE RESULTS")
    print("==================================")
    print(f"Input Code Length : {len(code_snippet)} characters")
    print(f"Prediction        : >> {prediction.upper()} <<")
    print("==================================\\n")

if __name__ == "__main__":
    # A sample C snippet to test the model
    sample_code = \"\"\"
    int main() {
        char buffer[10];
        strcpy(buffer, "This string is way too long for the buffer");
        return 0;
    }
    \"\"\"
    run_inference(sample_code)
"""

# 3. Write the script safely
with open(file_path, "w") as f:
    f.write(code_content)

# 4. Verify file generation
if os.path.exists(file_path):
    print(f"✅ Success! File physically exists at: {file_path}")
    print("Final contents of Dev/SRC directory:", os.listdir(base_dir))
else:
    print("❌ Error: File was not created.")

✅ Success! File physically exists at: /kaggle/working/Dev/SRC/inference.py
Final contents of Dev/SRC directory: ['inference.py']


In [3]:
!python3 /kaggle/working/Dev/SRC/inference.py

--- STEP 1: LOADING SAVED MODEL & TOKENIZER ---
❌ Error loading model: Repo id must be in the form 'repo_name' or 'namespace/repo_name': '/kaggle/working/best_model'. Use `repo_type` argument if needed.


In [4]:
import os

base_dir = "/kaggle/working/Dev/SRC"
file_path = os.path.join(base_dir, "inference.py")
os.makedirs(base_dir, exist_ok=True)

code_content = """import torch
import json
from transformers import AutoModelForSequenceClassification, AutoTokenizer

def run_inference(code_snippet):
    print("--- STEP 1: LOADING SAVED MODEL & TOKENIZER ---")
    
    # MLOps Best Practice: Pulling directly from your Hugging Face Hub
    # CHANGE THIS to your actual HF username and repo from Task 5
    hf_repo = "your-username/your-model-repo" 
    
    try:
        print(f"Pulling model from Hugging Face: {hf_repo}")
        model = AutoModelForSequenceClassification.from_pretrained(hf_repo)
        tokenizer = AutoTokenizer.from_pretrained(hf_repo)
    except Exception as e:
        print(f"❌ Error loading model: {e}\\nDid you replace 'your-username/your-model-repo'?")
        return

    print("\\n--- STEP 2: PREPARING INPUT DATA ---")
    cleaned_code = " ".join(code_snippet.split())
    
    inputs = tokenizer(
        cleaned_code,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=512
    )

    print("\\n--- STEP 3: RUNNING PREDICTION ---")
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        
    logits = outputs.logits
    predicted_class_id = logits.argmax(dim=1).item()
    
    id2label = {0: "Safe", 1: "Defective"}
    prediction = id2label[predicted_class_id]
    
    print("\\n==================================")
    print("      INFERENCE RESULTS")
    print("==================================")
    print(f"Input Code Length : {len(code_snippet)} characters")
    print(f"Prediction        : >> {prediction.upper()} <<")
    print("==================================\\n")

if __name__ == "__main__":
    sample_code = \"\"\"
    int main() {
        char buffer[10];
        strcpy(buffer, "This string is way too long for the buffer");
        return 0;
    }
    \"\"\"
    run_inference(sample_code)
"""

with open(file_path, "w") as f:
    f.write(code_content)

print("✅ inference.py updated to pull from Hugging Face Hub!")

✅ inference.py updated to pull from Hugging Face Hub!


In [5]:
!python3 /kaggle/working/Dev/SRC/inference.py

--- STEP 1: LOADING SAVED MODEL & TOKENIZER ---
Pulling model from Hugging Face: your-username/your-model-repo
❌ Error loading model: your-username/your-model-repo is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`
Did you replace 'your-username/your-model-repo'?


In [6]:
import os

file_path = "/kaggle/working/Dev/SRC/inference.py"

# 1. READ the current file
with open(file_path, "r") as f:
    code = f.read()

# 2. CHANGE THIS STRING to your actual Hugging Face repository!
my_real_hf_repo = "huggingface/CodeBERTa-small-v1" 

# 3. REPLACE the placeholder
code = code.replace("your-username/your-model-repo", my_real_hf_repo)

# 4. WRITE the fixed code back to the file
with open(file_path, "w") as f:
    f.write(code)

print(f"✅ Successfully updated inference.py to pull from: {my_real_hf_repo}")

✅ Successfully updated inference.py to pull from: huggingface/CodeBERTa-small-v1


In [7]:
!python3 /kaggle/working/Dev/SRC/inference.py

--- STEP 1: LOADING SAVED MODEL & TOKENIZER ---
Pulling model from Hugging Face: huggingface/CodeBERTa-small-v1
config.json: 100%|█████████████████████████████| 480/480 [00:00<00:00, 2.03MB/s]
pytorch_model.bin: 100%|█████████████████████| 336M/336M [00:06<00:00, 52.6MB/s]
Loading weights: 100%|█| 101/101 [00:00<00:00, 3463.93it/s, Materializing param=
RobertaForSequenceClassification LOAD REPORT from: huggingface/CodeBERTa-small-v1
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.decoder.bias        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.decoder.weight      | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSI

In [9]:
import os

# 1. Ensure the directory exists
base_dir = "/kaggle/working/Dev/SRC"
os.makedirs(base_dir, exist_ok=True)

# ---------------------------------------------------------
# FILE 1: data_preprocessing.py
# ---------------------------------------------------------
preprocessing_code = """import os
import json
from datasets import load_dataset
from transformers import AutoTokenizer

def clean_and_tokenize():
    dataset = load_dataset("google/code_x_glue_cc_defect_detection")
    model_name = "huggingface/CodeBERTa-small-v1"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    def preprocess_function(examples):
        cleaned_code = [" ".join(code.split()) for code in examples["func"]]
        model_inputs = tokenizer(cleaned_code, truncation=True, padding="max_length", max_length=512)
        model_inputs["labels"] = [1 if label else 0 for label in examples["target"]]
        return model_inputs

    columns_to_remove = ['id', 'func', 'target', 'project', 'commit_id']
    tokenized_dataset = dataset.map(preprocess_function, batched=True, remove_columns=columns_to_remove)
    
    output_dir = "/kaggle/working/processed_dataset"
    tokenized_dataset.save_to_disk(output_dir)
    
    with open("/kaggle/working/id2label.json", "w") as f:
        json.dump({0: "Safe", 1: "Defective"}, f)

if __name__ == "__main__":
    clean_and_tokenize()
"""
with open(os.path.join(base_dir, "data_preprocessing.py"), "w") as f:
    f.write(preprocessing_code)


# ---------------------------------------------------------
# FILE 2: train.py (With modern Hugging Face API fix)
# ---------------------------------------------------------
train_code = """import os
import json
import numpy as np
import evaluate
from datasets import load_from_disk
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding, AutoTokenizer
import wandb
from kaggle_secrets import UserSecretsClient

def train():
    try:
        secrets = UserSecretsClient()
        os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
        wandb.login()
    except:
        pass

    tokenized_dataset = load_from_disk("/kaggle/working/processed_dataset")
    model_name = "huggingface/CodeBERTa-small-v1"
    
    with open("/kaggle/working/id2label.json", "r") as f:
        id2label = {int(k): v for k, v in json.load(f).items()}
    
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2, id2label=id2label, label2id={v: k for k, v in id2label.items()})
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    accuracy_metric, f1_metric = evaluate.load("accuracy"), evaluate.load("f1")
    def compute_metrics(eval_pred):
        preds = np.argmax(eval_pred[0], axis=1)
        return {
            "eval_accuracy": accuracy_metric.compute(predictions=preds, references=eval_pred[1])["accuracy"],
            "eval_f1": f1_metric.compute(predictions=preds, references=eval_pred[1], average="binary")["f1"]
        }

    training_args = TrainingArguments(
        output_dir="/kaggle/working/results", learning_rate=3e-5, per_device_train_batch_size=16,
        num_train_epochs=3, eval_strategy="epoch", save_strategy="epoch", load_best_model_at_end=True,
        metric_for_best_model="eval_f1", fp16=True, report_to="wandb"
    )

    trainer = Trainer(
        model=model, args=training_args, train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["validation"], processing_class=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer), compute_metrics=compute_metrics,
    )
    trainer.train()
    trainer.save_model("/kaggle/working/best_model")

if __name__ == "__main__":
    train()
"""
with open(os.path.join(base_dir, "train.py"), "w") as f:
    f.write(train_code)


# ---------------------------------------------------------
# FILE 3: inference.py
# ---------------------------------------------------------
inference_code = """import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

def run_inference(code_snippet):
    # CHANGE THIS TO YOUR ACTUAL HUGGING FACE REPO URL
    hf_repo = "your-username/your-model-repo" 
    
    try:
        model = AutoModelForSequenceClassification.from_pretrained(hf_repo)
        tokenizer = AutoTokenizer.from_pretrained(hf_repo)
    except Exception as e:
        print(f"Error loading model: {e}")
        return

    cleaned_code = " ".join(code_snippet.split())
    inputs = tokenizer(cleaned_code, return_tensors="pt", truncation=True, padding="max_length", max_length=512)

    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        
    prediction = {0: "Safe", 1: "Defective"}[outputs.logits.argmax(dim=1).item()]
    print(f"\\nInput Code Length: {len(code_snippet)} | Prediction: >> {prediction.upper()} <<\\n")

if __name__ == "__main__":
    sample_code = "int main() { char buffer[10]; strcpy(buffer, 'too long'); return 0; }"
    run_inference(sample_code)
"""
with open(os.path.join(base_dir, "inference.py"), "w") as f:
    f.write(inference_code)

print("✅ All three files restored successfully!")
print(os.listdir(base_dir))

✅ All three files restored successfully!
['train.py', 'inference.py', 'data_preprocessing.py']


Tried to create standalone individual files

In [ ]:
%cd /kaggle/working
!ls

In [ ]:
!git config user.name "PrudhviGowroju"
!git config user.email "Prudhvi90hrithik@gmail.com"
!git config --list | grep user

In [ ]:
!git status


In [ ]:
!git checkout Prudhvi

In [ ]:
!git branch

In [ ]:
# 1. Install/upgrade necessary libraries
!pip install -q transformers datasets wandb evaluate scikit-learn

from kaggle_secrets import UserSecretsClient
import os
import wandb
from huggingface_hub import login
from datasets import load_dataset

# 2. Authenticate using Kaggle Secrets
try:
    secrets = UserSecretsClient()
    os.environ['WANDB_API_KEY'] = secrets.get_secret('WANDB_API_KEY')
    login(token=secrets.get_secret('HF_TOKEN'))
    wandb.login()
    print("✅ Successfully logged into Hugging Face and Weights & Biases!")
except Exception as e:
    print(f"⚠️ Secret loading error: {e}")

# 3. Load the dataset
print("\nDownloading the CodeXGLUE dataset...")
dataset = load_dataset("google/code_x_glue_cc_defect_detection")

# 4. Inspect the data structure
print("\n--- DATASET STRUCTURE ---")
print(dataset)

print("\n--- SAMPLE ITEM (Row 0) ---")
print(dataset['train'][0])

In [ ]:
import json
from transformers import AutoTokenizer

# 1. Load the tokenizer for CodeBERTa
model_name = "huggingface/CodeBERTa-small-v1"
print(f"Loading tokenizer for {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 2. Define the preprocessing function
def preprocess_function(examples):
    # Clean the code: replace multiple spaces/newlines with a single space
    cleaned_code = [" ".join(code.split()) for code in examples["func"]]
    
    # Tokenize the code (truncating to 512 tokens to fit BERT's maximum length)
    model_inputs = tokenizer(
        cleaned_code,
        truncation=True,
        padding="max_length",
        max_length=512
    )
    
    # Convert Boolean target to integers (0 = Safe, 1 = Defective)
    model_inputs["labels"] = [1 if label else 0 for label in examples["target"]]
    
    return model_inputs

# 3. Apply the processing to all splits and drop old columns
print("Cleaning, tokenizing, and dropping unused columns. This may take a minute...")
columns_to_remove = ['id', 'func', 'target', 'project', 'commit_id']
tokenized_dataset = dataset.map(
    preprocess_function, 
    batched=True, 
    remove_columns=columns_to_remove
)

# 4. Create and save the id2label mapping file
id2label = {0: "Safe", 1: "Defective"}

with open("id2label.json", "w") as f:
    json.dump(id2label, f)

print("✅ Data preparation complete! id2label.json is saved locally.")
print("\n--- TOKENIZED DATASET STRUCTURE ---")
print(tokenized_dataset)

In [ ]:
import wandb
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score

# 1. Load the Model
model_name = "huggingface/CodeBERTa-small-v1"
print(f"Loading {model_name} for sequence classification...")
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label={0: "Safe", 1: "Defective"},
    label2id={"Safe": 0, "Defective": 1}
)

# 2. Define Metrics (Accuracy and F1)
def compute_metrics(eval_pred):
    labels = eval_pred.label_ids
    preds = eval_pred.predictions.argmax(-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1': f1_score(labels, preds, average='weighted')
    }

# 3. Initialize W&B for Version 1
wandb.init(
    project="mlops-assignment3", 
    name="run-v1", 
    config={
        "model": model_name,
        "epochs": 3,
        "batch_size": 16,
        "learning_rate": 3e-5,
        "version": "v1",
        "platform": "Kaggle"
    }
)

# 4. Set Training Arguments
training_args = TrainingArguments(
    output_dir='./results_v1',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=3e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="wandb", 
    run_name="run-v1",
    fp16=True # Accelerates training on Kaggle GPUs
)

# 5. Initialize Trainer and Train
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['validation'],
    compute_metrics=compute_metrics
)

print("Starting training run v1...")
trainer.train()

# 6. Evaluate on Test Set and Finish W&B
print("\nEvaluating on test set...")
test_results = trainer.evaluate(tokenized_dataset['test'])
print(test_results)

wandb.finish()

In [ ]:
# 1. Initialize W&B for Version 2
wandb.init(
    project="mlops-assignment3", 
    name="run-v2", 
    config={
        "model": model_name,
        "epochs": 3,
        "batch_size": 16,
        "learning_rate": 5e-5, # HYPERPARAMETER CHANGE
        "version": "v2",
        "platform": "Kaggle"
    }
)

# 2. Set Training Arguments for v2
training_args_v2 = TrainingArguments(
    output_dir='./results_v2',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=5e-5, # HYPERPARAMETER CHANGE
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="wandb", 
    run_name="run-v2",
    fp16=True 
)

# 3. Initialize Trainer and Train
trainer_v2 = Trainer(
    model=model,
    args=training_args_v2,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['validation'],
    compute_metrics=compute_metrics
)

print("Starting training run v2...")
trainer_v2.train()

# 4. Evaluate on Test Set and Finish W&B
print("\nEvaluating on test set for run v2...")
test_results_v2 = trainer_v2.evaluate(tokenized_dataset['test'])
print(test_results_v2)

wandb.finish()